### Cell 12.01 — Reconstruct manuscript Table 2: four suggestive QTL

In [ ]:
# Cell 12.01 — corrected
# Reconstruct manuscript Table 2 from frozen all-trait QTL results

from pathlib import Path
import pandas as pd
import numpy as np

def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from within "
        "the flyer-hartwig-qtl-reanalysis repository."
    )

PROJECT_ROOT = find_project_root()

QTL_FILE = PROJECT_ROOT / "results/qtl/flyer_hartwig_all33_empirical_qtl_screen.xlsx"
TABLE_DIR = PROJECT_ROOT / "results/tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

qtl = pd.read_excel(QTL_FILE, sheet_name="all_33_empirical")

print("Columns:")
print(qtl.columns.tolist())

# Keep only the four genome-wide suggestive loci
table2 = qtl.loc[
    qtl["genomewide_status"].eq("suggestive_10pct")
].copy()

print("\nSuggestive loci found:", len(table2))
display(table2)

### Cell 12.02 — corrected Format manuscript-ready Table 2

In [ ]:
# Cell 12.02 — corrected
# Format manuscript-ready Table 2

physical_resolution = {
    "scn_fi3": "Two-sided anchor-defined physical bracket",
    "lrn": "Two-sided anchor-defined physical bracket",
    "prot_03": "Exact peak-marker anchor only",
    "days_fl_07": "Single-anchor chromosome assignment only",
}

table2_final = pd.DataFrame({
    "Trait": table2["trait"],
    "Peak marker": table2["peak_marker"],
    "Linkage fragment": table2["structural_group"],
    "Chr": table2["candidate_chr"],
    "N": table2["n"],
    "LOD": table2["lod"],
    "R²": table2["r2"],
    "Empirical P": table2["peak_empirical_p"],
})

# We will merge the frozen allele-effect values separately below
effect_values = {
    "scn_fi3": -29.0,   # temporary placeholder; replace from frozen allele-effect file
    "prot_03": np.nan,
    "days_fl_07": -3.344615,
    "lrn": np.nan,
}

table2_final["Flyer−Hartwig effect"] = (
    table2_final["Trait"].map(effect_values)
)

table2_final["Physical resolution"] = (
    table2_final["Trait"].map(physical_resolution)
)

# Preferred manuscript order
trait_order = ["scn_fi3", "prot_03", "days_fl_07", "lrn"]

table2_final["__order"] = pd.Categorical(
    table2_final["Trait"],
    categories=trait_order,
    ordered=True
)

table2_final = (
    table2_final
    .sort_values("__order")
    .drop(columns="__order")
    .reset_index(drop=True)
)

# Manuscript-friendly precision
table2_final["LOD"] = table2_final["LOD"].astype(float).round(3)
table2_final["R²"] = table2_final["R²"].astype(float).round(3)
table2_final["Empirical P"] = table2_final["Empirical P"].astype(float).round(3)
table2_final["Flyer−Hartwig effect"] = (
    table2_final["Flyer−Hartwig effect"].astype(float).round(3)
)

display(table2_final)

### Cell 12.03 - Search the QTL workbook sheets for allele-effect columns

In [ ]:
# Cell 12.03
# Search the QTL workbook sheets for allele-effect columns

xls = pd.ExcelFile(QTL_FILE)

print("Sheets:")
print(xls.sheet_names)

for sheet in xls.sheet_names:
    df = pd.read_excel(QTL_FILE, sheet_name=sheet)
    effect_cols = [
        c for c in df.columns
        if "effect" in str(c).lower()
        or "mean0" in str(c).lower()
        or "mean2" in str(c).lower()
    ]

    if effect_cols:
        print(f"\nSheet: {sheet}")
        print("Possible effect columns:", effect_cols)
        print("All columns:", df.columns.tolist())

### Cell 12.04 — Extract exact Flyer−Hartwig effects from the four frozen regional profiles

In [ ]:
# Cell 12.04
# Recover exact peak-marker allele effects from frozen regional QTL sheets

region_sheets = {
    "scn_fi3": "region_scn_fi3",
    "prot_03": "region_prot_03",
    "days_fl_07": "region_days_fl_07",
    "lrn": "region_lrn",
}

peak_markers = {
    "scn_fi3": "Satt354",
    "prot_03": "Satt440",
    "days_fl_07": "TMA2",
    "lrn": "Satt282a",
}

effect_rows = []

for trait, sheet in region_sheets.items():
    df = pd.read_excel(QTL_FILE, sheet_name=sheet)

    peak_marker = peak_markers[trait]

    hit = df.loc[
        df["marker"].astype(str).eq(peak_marker)
    ].copy()

    if len(hit) != 1:
        raise ValueError(
            f"{trait}: expected exactly 1 row for {peak_marker}, found {len(hit)}"
        )

    row = hit.iloc[0]

    effect_rows.append({
        "Trait": trait,
        "Peak marker": peak_marker,
        "N_effect_sheet": int(row["n"]),
        "Hartwig mean (0)": row["mean_genotype_0"],
        "Flyer mean (2)": row["mean_genotype_2"],
        "Flyer−Hartwig effect": row["effect_2_minus_0"],
        "Peak LOD check": row["lod"],
    })

peak_effects = pd.DataFrame(effect_rows)

display(peak_effects)

### Cell 12.05 — Build the final manuscript Table 2 and validate it

In [ ]:
# Cell 12.05
# Final manuscript Table 2 using only frozen values

physical_resolution = {
    "scn_fi3": "Two-sided anchor-defined physical bracket",
    "prot_03": "Exact peak-marker anchor only",
    "days_fl_07": "Single-anchor chromosome assignment only",
    "lrn": "Two-sided anchor-defined physical bracket",
}

# Start from the four empirically suggestive loci
table2_final = table2[
    [
        "trait",
        "peak_marker",
        "structural_group",
        "candidate_chr",
        "n",
        "lod",
        "r2",
        "peak_empirical_p",
    ]
].copy()

table2_final = table2_final.rename(columns={
    "trait": "Trait",
    "peak_marker": "Peak marker",
    "structural_group": "Linkage fragment",
    "candidate_chr": "Chr",
    "n": "N",
    "lod": "LOD",
    "r2": "R²",
    "peak_empirical_p": "Empirical P",
})

# Merge exact effect values from frozen regional profiles
table2_final = table2_final.merge(
    peak_effects[
        ["Trait", "Peak marker", "Flyer−Hartwig effect"]
    ],
    on=["Trait", "Peak marker"],
    how="left",
    validate="one_to_one"
)

table2_final["Physical resolution"] = (
    table2_final["Trait"].map(physical_resolution)
)

# Preferred manuscript order
trait_order = [
    "scn_fi3",
    "prot_03",
    "days_fl_07",
    "lrn",
]

table2_final["__order"] = pd.Categorical(
    table2_final["Trait"],
    categories=trait_order,
    ordered=True
)

table2_final = (
    table2_final
    .sort_values("__order")
    .drop(columns="__order")
    .reset_index(drop=True)
)

# Validate against frozen peak rows
assert len(table2_final) == 4
assert table2_final["Flyer−Hartwig effect"].notna().all()
assert table2_final["Peak marker"].nunique() == 4

# Display with manuscript-friendly precision
table2_display = table2_final.copy()

table2_display["LOD"] = table2_display["LOD"].round(3)
table2_display["R²"] = table2_display["R²"].round(3)
table2_display["Empirical P"] = table2_display["Empirical P"].round(3)
table2_display["Flyer−Hartwig effect"] = (
    table2_display["Flyer−Hartwig effect"].round(3)
)

display(table2_display)

### Cell 12.06 — Save Table 2 in manuscript-ready formats

In [ ]:
# Cell 12.06
# Save final Table 2

TABLE2_CSV = TABLE_DIR / "Table2_four_suggestive_qtl.csv"
TABLE2_XLSX = TABLE_DIR / "Table2_four_suggestive_qtl.xlsx"

# Save full numerical precision
table2_final.to_csv(TABLE2_CSV, index=False)

with pd.ExcelWriter(TABLE2_XLSX, engine="openpyxl") as writer:
    table2_final.to_excel(
        writer,
        sheet_name="Table2_suggestive_QTL",
        index=False
    )

print("Saved final Table 2:")
print(TABLE2_CSV)
print(TABLE2_XLSX)

print("\nFinal manuscript display:")
display(table2_display)

### Cell 12.07 — Locate and inspect the frozen SCN candidate workbook

In [ ]:
# Cell 12.07
# Locate the frozen SCN Gm20 candidate-gene workbook

from pathlib import Path
import pandas as pd
import numpy as np

CAND_DIR = PROJECT_ROOT / "results/candidate_genes"

candidate_files = [
    CAND_DIR / "flyer_hartwig_scn_gm20_final_evidence_summary.xlsx",
    CAND_DIR / "flyer_hartwig_scn_gm20_literature_integrated_candidates.xlsx",
    CAND_DIR / "flyer_hartwig_scn_gm20_candidate_prioritization.xlsx",
]

SCN_FILE = next((p for p in candidate_files if p.exists()), None)

if SCN_FILE is None:
    raise FileNotFoundError(
        "No frozen SCN Gm20 candidate workbook was found."
    )

print("Using workbook:")
print(SCN_FILE)

xls = pd.ExcelFile(SCN_FILE)

print("\nSheets:")
print(xls.sheet_names)

### Cell 12.08 — Load the nine high-priority genes and inspect columns

In [ ]:
# Cell 12.08
# Load the frozen nine-gene high-priority SCN candidate set

preferred_sheets = [
    "high_priority_9",
    "final_candidates",
    "evidence_shortlist",
    "all_ranked_24",
]

sheet = next(
    (s for s in preferred_sheets if s in xls.sheet_names),
    None
)

if sheet is None:
    raise KeyError(
        f"Could not identify the high-priority candidate sheet.\n"
        f"Available sheets: {xls.sheet_names}"
    )

scn_candidates = pd.read_excel(
    SCN_FILE,
    sheet_name=sheet
)

print("Using sheet:", sheet)
print("\nColumns:")
print(scn_candidates.columns.tolist())

display(scn_candidates)

### Cell 12.09 — Reconstruct manuscript Table 3
* This uses the exact frozen nine-gene list and adds the manuscript-level evidence interpretation.

In [ ]:
# Cell 12.09
# Reconstruct manuscript Table 3 from frozen SCN candidate results

HIGH_PRIORITY_GENES = [
    "Glyma.20G078700",
    "Glyma.20G078800",
    "Glyma.20G080700",
    "Glyma.20G083000",
    "Glyma.20G100000",
    "Glyma.20G100500",
    "Glyma.20G101100",
    "Glyma.20G104000",
    "Glyma.20G111300",
]

def find_col(df, candidates):
    """Find a likely column by exact or partial case-insensitive match."""
    lower_map = {
        str(col).strip().lower(): col
        for col in df.columns
    }

    # Exact match first
    for wanted in candidates:
        if wanted.lower() in lower_map:
            return lower_map[wanted.lower()]

    # Partial match second
    for col in df.columns:
        col_lower = str(col).lower()
        for wanted in candidates:
            if wanted.lower() in col_lower:
                return col

    return None


gene_col = find_col(
    scn_candidates,
    ["gene", "gene_id", "name", "id"]
)

start_col = find_col(
    scn_candidates,
    ["start_bp", "start"]
)

end_col = find_col(
    scn_candidates,
    ["end_bp", "end"]
)

annotation_col = find_col(
    scn_candidates,
    [
        "functional_annotation",
        "gene_title",
        "annotation",
        "note"
    ]
)

evidence_col = find_col(
    scn_candidates,
    [
        "evidence_class",
        "priority_class",
        "tier"
    ]
)

print("Detected columns:")
print("gene:", gene_col)
print("start:", start_col)
print("end:", end_col)
print("annotation:", annotation_col)
print("evidence:", evidence_col)

if gene_col is None:
    raise KeyError("Could not identify gene column.")

if start_col is None or end_col is None:
    raise KeyError("Could not identify start/end coordinate columns.")

if annotation_col is None:
    raise KeyError("Could not identify functional annotation column.")

# Keep exactly the frozen nine high-priority genes

tmp = scn_candidates.copy()

def normalize_gene_name(value):
    text = str(value)

    for gene in HIGH_PRIORITY_GENES:
        if gene in text:
            return gene

    return text


tmp["Gene_clean"] = tmp[gene_col].map(
    normalize_gene_name
)

table3_source = tmp.loc[
    tmp["Gene_clean"].isin(HIGH_PRIORITY_GENES)
].copy()

if table3_source["Gene_clean"].nunique() != 9:
    raise ValueError(
        "Expected 9 unique high-priority genes, "
        f"found {table3_source['Gene_clean'].nunique()}."
    )

# Convert bp -> Mb
table3_source["Start Mb"] = (
    pd.to_numeric(
        table3_source[start_col],
        errors="coerce"
    ) / 1_000_000
)

table3_source["End Mb"] = (
    pd.to_numeric(
        table3_source[end_col],
        errors="coerce"
    ) / 1_000_000
)

# Frozen evidence classification from Notebook 06

direct_scn_evidence = {
    "Glyma.20G078700": "No",
    "Glyma.20G078800": "No",
    "Glyma.20G080700": "No",
    "Glyma.20G083000": "No",
    "Glyma.20G100000": "No",
    "Glyma.20G100500": "No",
    "Glyma.20G101100": "No",
    "Glyma.20G104000": "Yes",
    "Glyma.20G111300": "No",
}

frozen_evidence_class = {
    "Glyma.20G078700":
        "Tier 1B — strong functional candidate",

    "Glyma.20G078800":
        "Tier 1B — strong functional candidate",

    "Glyma.20G080700":
        "Tier 1B — strong functional candidate",

    "Glyma.20G083000":
        "Tier 1B — strong functional candidate",

    "Glyma.20G100000":
        "Tier 1B — strong functional candidate",

    "Glyma.20G100500":
        "Tier 1B — strong functional candidate",

    "Glyma.20G101100":
        "Tier 1B — strong functional candidate",

    "Glyma.20G104000":
        "Tier 1A — direct SCN molecular evidence",

    "Glyma.20G111300":
        "Tier 1B — strong functional candidate",
}


interpretation = {
    "Glyma.20G078700":
        "Disease-responsive dirigent-like candidate; plausible role in defense-associated cell-wall or phenylpropanoid responses.",

    "Glyma.20G078800":
        "Disease-responsive dirigent-like candidate; plausible role in defense-associated cell-wall or phenylpropanoid responses.",

    "Glyma.20G080700":
        "Receptor-like kinase candidate with plausible involvement in pathogen perception or defense signaling.",

    "Glyma.20G083000":
        "Cysteine-rich receptor-like kinase candidate; plausible defense-signaling function.",

    "Glyma.20G100000":
        "Thioredoxin-related candidate; plausible role in redox regulation during stress responses.",

    "Glyma.20G100500":
        "Receptor-like kinase candidate with independent soybean biotic-stress context, but no direct SCN validation.",

    "Glyma.20G101100":
        "Glutathione S-transferase family candidate; plausible detoxification and redox-defense function.",

    "Glyma.20G104000":
        "Highest-evidence candidate; independent SCN-responsive molecular evidence, but no demonstrated causal resistance.",

    "Glyma.20G111300":
        "Hemerythrin-class glutathione S-transferase candidate; plausible stress-response and detoxification function.",
}

# Build final Table 3

table3_final = pd.DataFrame({
    "Gene":
        table3_source["Gene_clean"],

    "Start Mb":
        table3_source["Start Mb"],

    "End Mb":
        table3_source["End Mb"],

    "Functional annotation":
        table3_source[annotation_col],
})

# Prefer our frozen standardized evidence labels
table3_final["Evidence class"] = (
    table3_final["Gene"]
    .map(frozen_evidence_class)
)

table3_final["Direct SCN evidence"] = (
    table3_final["Gene"]
    .map(direct_scn_evidence)
)

table3_final["Interpretation"] = (
    table3_final["Gene"]
    .map(interpretation)
)

# Sort by physical coordinate
table3_final = (
    table3_final
    .sort_values("Start Mb")
    .reset_index(drop=True)
)

# Manuscript precision
table3_display = table3_final.copy()

table3_display["Start Mb"] = (
    table3_display["Start Mb"]
    .round(6)
)

table3_display["End Mb"] = (
    table3_display["End Mb"]
    .round(6)
)

# Validate
assert len(table3_final) == 9
assert table3_final["Gene"].nunique() == 9
assert table3_final["Start Mb"].notna().all()
assert table3_final["End Mb"].notna().all()
assert table3_final["Functional annotation"].notna().all()

display(table3_display)

### Cell 12.10 — Save Table 3

In [ ]:
# Cell 12.10
# Save manuscript Table 3

TABLE3_CSV = (
    TABLE_DIR
    / "Table3_scn_gm20_high_priority_candidate_genes.csv"
)

TABLE3_XLSX = (
    TABLE_DIR
    / "Table3_scn_gm20_high_priority_candidate_genes.xlsx"
)

# Full precision CSV
table3_final.to_csv(
    TABLE3_CSV,
    index=False
)

# Excel manuscript table
with pd.ExcelWriter(
    TABLE3_XLSX,
    engine="openpyxl"
) as writer:

    table3_final.to_excel(
        writer,
        sheet_name="Table3_SCN_candidates",
        index=False
    )

print("Saved Table 3:")
print(TABLE3_CSV)
print(TABLE3_XLSX)

display(table3_display)

### Cell 12.11 — Set up supplementary-table output and load frozen sources

In [ ]:
# Cell 12.11
# Supplementary Tables S1-S10
# Setup + frozen source loading

from pathlib import Path
import pandas as pd
import numpy as np

SUPP_TABLE_DIR = TABLE_DIR / "supplementary_tables"
SUPP_TABLE_DIR.mkdir(parents=True, exist_ok=True)

MAP_FILE = (
    PROJECT_ROOT
    / "results/linkage_map/flyer_hartwig_structural_physical_map_final.xlsx"
)

QTL_FILE = (
    PROJECT_ROOT
    / "results/qtl/flyer_hartwig_all33_empirical_qtl_screen.xlsx"
)

print("Frozen files:")
print("Map:", MAP_FILE.exists(), MAP_FILE)
print("QTL:", QTL_FILE.exists(), QTL_FILE)

print("\nMap sheets:")
print(pd.ExcelFile(MAP_FILE).sheet_names)

print("\nQTL sheets:")
print(pd.ExcelFile(QTL_FILE).sheet_names)

print("\nSupplementary-table directory:")
print(SUPP_TABLE_DIR)

## Supplementary Table S1
### Complete empirical QTL results for all 33 traits
### Cell 12.12 — Build and save Table S1

In [ ]:
# Cell 12.12
# Supplementary Table S1
# Complete genome-wide empirical QTL results for all 33 traits

s1 = pd.read_excel(
    QTL_FILE,
    sheet_name="all_33_empirical"
).copy()

# Publication-friendly column names
s1 = s1.rename(columns={
    "trait": "Trait",
    "peak_marker": "Peak marker",
    "structural_group": "Linkage fragment",
    "candidate_chr": "Chr",
    "n": "N",
    "r2": "R²",
    "lod": "LOD",
    "lod_threshold_10pct": "10% LOD threshold",
    "lod_threshold_05pct": "5% LOD threshold",
    "lod_threshold_01pct": "1% LOD threshold",
    "peak_empirical_p": "Empirical P",
    "genomewide_status": "Genome-wide status",
})

# Sort primarily by decreasing peak LOD
s1 = (
    s1
    .sort_values("LOD", ascending=False)
    .reset_index(drop=True)
)

# Display copy with manuscript precision
s1_display = s1.copy()

for c in [
    "R²",
    "LOD",
    "10% LOD threshold",
    "5% LOD threshold",
    "1% LOD threshold",
    "Empirical P",
]:
    if c in s1_display.columns:
        s1_display[c] = pd.to_numeric(
            s1_display[c],
            errors="coerce"
        ).round(3)

display(s1_display)

# Save exact numerical table
S1_CSV = SUPP_TABLE_DIR / "TableS1_all33_empirical_qtl_results.csv"
S1_XLSX = SUPP_TABLE_DIR / "TableS1_all33_empirical_qtl_results.xlsx"

s1.to_csv(S1_CSV, index=False)

with pd.ExcelWriter(S1_XLSX, engine="openpyxl") as writer:
    s1.to_excel(
        writer,
        sheet_name="TableS1",
        index=False
    )

print("\nSaved Table S1:")
print(S1_CSV)
print(S1_XLSX)

### Cell 12.13 — Build and save Table S2

In [ ]:
# Cell 12.13
# Supplementary Table S2
# Final structurally corrected linkage-fragment summary

s2 = pd.read_excel(
    MAP_FILE,
    sheet_name="group_summary"
).copy()

print("Original columns:")
print(s2.columns.tolist())

# Flexible publication-friendly renaming
rename_s2 = {
    "structural_group": "Linkage fragment",
    "group": "Linkage fragment",
    "n_markers": "Ordered markers",
    "marker_count": "Ordered markers",
    "n_intervals": "Retained intervals",
    "kosambi_length_cm": "Kosambi length (cM)",
    "kosambi_cm": "Kosambi length (cM)",
    "haldane_length_cm": "Haldane length (cM)",
    "haldane_cm": "Haldane length (cM)",
    "dominant_chr": "Candidate chromosome",
    "candidate_chr": "Candidate chromosome",
    "assignment_status": "Physical assignment confidence",
}

s2 = s2.rename(
    columns={
        k: v
        for k, v in rename_s2.items()
        if k in s2.columns
    }
)

# Put the most useful columns first, retain any additional frozen fields
preferred_s2 = [
    "Linkage fragment",
    "Ordered markers",
    "Retained intervals",
    "Kosambi length (cM)",
    "Haldane length (cM)",
    "Candidate chromosome",
    "Physical assignment confidence",
]

front = [c for c in preferred_s2 if c in s2.columns]
rest = [c for c in s2.columns if c not in front]

s2 = s2[front + rest].copy()

# Manuscript display precision
s2_display = s2.copy()

for c in [
    "Kosambi length (cM)",
    "Haldane length (cM)",
]:
    if c in s2_display.columns:
        s2_display[c] = pd.to_numeric(
            s2_display[c],
            errors="coerce"
        ).round(3)

display(s2_display)

S2_CSV = SUPP_TABLE_DIR / "TableS2_structural_linkage_fragment_summary.csv"
S2_XLSX = SUPP_TABLE_DIR / "TableS2_structural_linkage_fragment_summary.xlsx"

s2.to_csv(S2_CSV, index=False)

with pd.ExcelWriter(S2_XLSX, engine="openpyxl") as writer:
    s2.to_excel(
        writer,
        sheet_name="TableS2",
        index=False
    )

print("\nSaved Table S2:")
print(S2_CSV)
print(S2_XLSX)

### Cell 12.14 — Build and save Tables S3 and S4
* We can efficiently make the next two in one cell.

In [ ]:
# Cell 12.14
# Supplementary Tables S3 and S4
# S3 = independent physical assignments
# S4 = removed structural joins

# ============================================================
# TABLE S3
# ============================================================

s3 = pd.read_excel(
    MAP_FILE,
    sheet_name="physical_assignments"
).copy()

print("S3 original columns:")
print(s3.columns.tolist())

rename_s3 = {
    "structural_group": "Linkage fragment",
    "group": "Linkage fragment",
    "dominant_chr": "Assigned chromosome",
    "candidate_chr": "Assigned chromosome",
    "assignment_status": "Assignment confidence",
    "n_independent_assays": "Independent assays",
    "n_exact": "Exact anchors",
    "n_family": "Family-level anchors",
    "n_family_matches": "Family-level anchors",
}

s3 = s3.rename(
    columns={
        k: v
        for k, v in rename_s3.items()
        if k in s3.columns
    }
)

preferred_s3 = [
    "Linkage fragment",
    "Assigned chromosome",
    "Assignment confidence",
    "Independent assays",
    "Exact anchors",
    "Family-level anchors",
]

front = [c for c in preferred_s3 if c in s3.columns]
rest = [c for c in s3.columns if c not in front]

s3 = s3[front + rest].copy()

display(s3)

S3_CSV = SUPP_TABLE_DIR / "TableS3_independent_physical_assignments.csv"
S3_XLSX = SUPP_TABLE_DIR / "TableS3_independent_physical_assignments.xlsx"

s3.to_csv(S3_CSV, index=False)

with pd.ExcelWriter(S3_XLSX, engine="openpyxl") as writer:
    s3.to_excel(
        writer,
        sheet_name="TableS3",
        index=False
    )


# ============================================================
# TABLE S4
# ============================================================

s4 = pd.read_excel(
    MAP_FILE,
    sheet_name="removed_joins"
).copy()

print("\nS4 original columns:")
print(s4.columns.tolist())

rename_s4 = {
    "structural_group": "Original linkage group",
    "group": "Original linkage group",
    "marker1": "Marker 1",
    "marker2": "Marker 2",
    "marker_a": "Marker 1",
    "marker_b": "Marker 2",
    "n": "Informative RILs",
    "r": "Recombination fraction",
    "R": "Recombination fraction",
    "lod": "Pairwise LOD",
    "kosambi_cm": "Removed interval (cM)",
    "kosambi_distance_cm": "Removed interval (cM)",
}

s4 = s4.rename(
    columns={
        k: v
        for k, v in rename_s4.items()
        if k in s4.columns
    }
)

preferred_s4 = [
    "Original linkage group",
    "Marker 1",
    "Marker 2",
    "Informative RILs",
    "Recombination fraction",
    "Pairwise LOD",
    "Removed interval (cM)",
]

front = [c for c in preferred_s4 if c in s4.columns]
rest = [c for c in s4.columns if c not in front]

s4 = s4[front + rest].copy()

s4_display = s4.copy()

for c in [
    "Recombination fraction",
    "Pairwise LOD",
    "Removed interval (cM)",
]:
    if c in s4_display.columns:
        s4_display[c] = pd.to_numeric(
            s4_display[c],
            errors="coerce"
        ).round(6)

display(s4_display)

S4_CSV = SUPP_TABLE_DIR / "TableS4_removed_structural_joins.csv"
S4_XLSX = SUPP_TABLE_DIR / "TableS4_removed_structural_joins.xlsx"

s4.to_csv(S4_CSV, index=False)

with pd.ExcelWriter(S4_XLSX, engine="openpyxl") as writer:
    s4.to_excel(
        writer,
        sheet_name="TableS4",
        index=False
    )

print("\nSaved:")
print(S3_CSV)
print(S3_XLSX)
print(S4_CSV)
print(S4_XLSX)

### Cell 12.15 — Supplementary Tables S5 and S6

In [ ]:
# Cell 12.15
# Supplementary Tables S5 and S6
# S5 = peak allele effects for the four suggestive QTL
# S6 = physical/candidate evidence summary for the four suggestive QTL

# ============================================================
# TABLE S5
# ============================================================

region_sheets = {
    "scn_fi3": "region_scn_fi3",
    "prot_03": "region_prot_03",
    "days_fl_07": "region_days_fl_07",
    "lrn": "region_lrn",
}

peak_markers = {
    "scn_fi3": "Satt354",
    "prot_03": "Satt440",
    "days_fl_07": "TMA2",
    "lrn": "Satt282a",
}

s5_rows = []

for trait, sheet in region_sheets.items():
    df = pd.read_excel(QTL_FILE, sheet_name=sheet)

    hit = df.loc[
        df["marker"].astype(str).eq(peak_markers[trait])
    ].copy()

    if len(hit) != 1:
        raise ValueError(
            f"{trait}: expected 1 peak row for {peak_markers[trait]}, "
            f"found {len(hit)}"
        )

    r = hit.iloc[0]

    s5_rows.append({
        "Trait": trait,
        "Peak marker": peak_markers[trait],
        "Linkage fragment": r["structural_group"],
        "Chr": r["dominant_chr"],
        "N": r["n"],
        "Hartwig mean (genotype 0)": r["mean_genotype_0"],
        "Flyer mean (genotype 2)": r["mean_genotype_2"],
        "Flyer−Hartwig effect": r["effect_2_minus_0"],
        "R²": r["r2"],
        "LOD": r["lod"],
    })

s5 = pd.DataFrame(s5_rows)

trait_order = ["scn_fi3", "prot_03", "days_fl_07", "lrn"]

s5["__order"] = pd.Categorical(
    s5["Trait"],
    categories=trait_order,
    ordered=True
)

s5 = (
    s5
    .sort_values("__order")
    .drop(columns="__order")
    .reset_index(drop=True)
)

s5_display = s5.copy()

for c in [
    "Hartwig mean (genotype 0)",
    "Flyer mean (genotype 2)",
    "Flyer−Hartwig effect",
    "R²",
    "LOD",
]:
    s5_display[c] = pd.to_numeric(
        s5_display[c],
        errors="coerce"
    ).round(3)

display(s5_display)

S5_CSV = SUPP_TABLE_DIR / "TableS5_suggestive_qtl_peak_allele_effects.csv"
S5_XLSX = SUPP_TABLE_DIR / "TableS5_suggestive_qtl_peak_allele_effects.xlsx"

s5.to_csv(S5_CSV, index=False)

with pd.ExcelWriter(S5_XLSX, engine="openpyxl") as writer:
    s5.to_excel(writer, sheet_name="TableS5", index=False)


# ============================================================
# TABLE S6
# ============================================================

s6 = pd.DataFrame([
    {
        "Trait": "scn_fi3",
        "Peak marker": "Satt354",
        "Linkage fragment": "pLG01b_Gm20",
        "Chr": "Gm20",
        "Physical assignment confidence": "supported",
        "Physical resolution": "Two-sided anchor-defined physical bracket",
        "Region start (Mb)": 28.303434,
        "Region end (Mb)": 38.482498,
        "Peak direct anchor": "No",
        "Candidate-gene analysis": "Yes",
        "Candidate evidence summary":
            "307 genes in bracket; 24 ranked candidates; 9 high-priority genes; "
            "Glyma.20G104000 has direct SCN-responsive molecular evidence.",
        "Interpretation":
            "Best disease-related suggestive region; physical bracket is not a statistical QTL confidence interval."
    },
    {
        "Trait": "lrn",
        "Peak marker": "Satt282a",
        "Linkage fragment": "pLG04",
        "Chr": "Gm02",
        "Physical assignment confidence": "strong",
        "Physical resolution": "Two-sided anchor-defined physical bracket",
        "Region start (Mb)": 20.165739,
        "Region end (Mb)": 46.904614,
        "Peak direct anchor": "Assay-family anchor at 30.354248 Mb",
        "Candidate-gene analysis": "Yes",
        "Candidate evidence summary":
            "429 genes in bracket; 13 conservative mechanistic candidates; "
            "8 evidence-aware literature shortlist genes.",
        "Interpretation":
            "Broad physical bracket; candidate proximity or density was not used to narrow the QTL region."
    },
    {
        "Trait": "prot_03",
        "Peak marker": "Satt440",
        "Linkage fragment": "pLG11",
        "Chr": "Gm20",
        "Physical assignment confidence": "supported",
        "Physical resolution": "Exact peak-marker anchor only",
        "Region start (Mb)": np.nan,
        "Region end (Mb)": np.nan,
        "Peak direct anchor": "Exact anchor at 49.904048 Mb",
        "Candidate-gene analysis": "Descriptive only",
        "Candidate evidence summary":
            "Anchor-centered ±1 Mb descriptive context; 19 curated candidates after false-positive removal.",
        "Interpretation":
            "No closed physical QTL bracket; anchor-centered windows are not statistical confidence intervals."
    },
    {
        "Trait": "days_fl_07",
        "Peak marker": "TMA2",
        "Linkage fragment": "pLG07",
        "Chr": "Gm08",
        "Physical assignment confidence": "insufficient_single_anchor",
        "Physical resolution": "Single-anchor chromosome assignment only",
        "Region start (Mb)": np.nan,
        "Region end (Mb)": np.nan,
        "Peak direct anchor": "No; Sat_162 anchors fragment at 8.327526 Mb",
        "Candidate-gene analysis": "No",
        "Candidate evidence summary":
            "No defensible physical interval or candidate-gene window.",
        "Interpretation":
            "TMA2 lies 2.742 provisional cM from Sat_162; genetic distance was not converted to physical distance."
    },
])

display(s6)

S6_CSV = SUPP_TABLE_DIR / "TableS6_suggestive_qtl_physical_candidate_evidence.csv"
S6_XLSX = SUPP_TABLE_DIR / "TableS6_suggestive_qtl_physical_candidate_evidence.xlsx"

s6.to_csv(S6_CSV, index=False)

with pd.ExcelWriter(S6_XLSX, engine="openpyxl") as writer:
    s6.to_excel(writer, sheet_name="TableS6", index=False)

print("\nSaved:")
print(S5_CSV)
print(S5_XLSX)
print(S6_CSV)
print(S6_XLSX)

### Cell 12.16 — Supplementary Tables S7 and S8

In [ ]:
# Cell 12.16
# Supplementary Tables S7 and S8
# S7 = complete ranked SCN Gm20 candidate set
# S8 = all genes within the SCN Gm20 bracket

CAND_DIR = PROJECT_ROOT / "results/candidate_genes"

SCN_FINAL_FILE = (
    CAND_DIR
    / "flyer_hartwig_scn_gm20_final_evidence_summary.xlsx"
)

if not SCN_FINAL_FILE.exists():
    raise FileNotFoundError(
        f"Missing frozen SCN final workbook:\n{SCN_FINAL_FILE}"
    )

print("SCN workbook sheets:")
print(pd.ExcelFile(SCN_FINAL_FILE).sheet_names)

# ============================================================
# TABLE S7
# ============================================================

s7 = pd.read_excel(
    SCN_FINAL_FILE,
    sheet_name="all_ranked_24"
).copy()

display(s7.head())
print("S7 rows:", len(s7))

S7_CSV = SUPP_TABLE_DIR / "TableS7_scn_gm20_ranked_24_candidates.csv"
S7_XLSX = SUPP_TABLE_DIR / "TableS7_scn_gm20_ranked_24_candidates.xlsx"

s7.to_csv(S7_CSV, index=False)

with pd.ExcelWriter(S7_XLSX, engine="openpyxl") as writer:
    s7.to_excel(writer, sheet_name="TableS7", index=False)


# ============================================================
# TABLE S8
# ============================================================

s8 = pd.read_excel(
    SCN_FINAL_FILE,
    sheet_name="all_region_307"
).copy()

display(s8.head())
print("S8 rows:", len(s8))

S8_CSV = SUPP_TABLE_DIR / "TableS8_scn_gm20_all_307_genes.csv"
S8_XLSX = SUPP_TABLE_DIR / "TableS8_scn_gm20_all_307_genes.xlsx"

s8.to_csv(S8_CSV, index=False)

with pd.ExcelWriter(S8_XLSX, engine="openpyxl") as writer:
    s8.to_excel(writer, sheet_name="TableS8", index=False)

print("\nSaved:")
print(S7_CSV)
print(S7_XLSX)
print(S8_CSV)
print(S8_XLSX)

### Cell 12.17 — Supplementary Tables S9 and S10

In [ ]:
# Cell 12.17
# Supplementary Tables S9 and S10
# S9 = LRN Gm02 evidence-aware shortlist
# S10 = prot_03 Gm20 curated anchor-centered candidate set

# ============================================================
# TABLE S9
# ============================================================

LRN_FILE = (
    CAND_DIR
    / "flyer_hartwig_lrn_gm02_final_candidates.xlsx"
)

if not LRN_FILE.exists():
    raise FileNotFoundError(
        f"Missing frozen LRN workbook:\n{LRN_FILE}"
    )

print("LRN workbook sheets:")
print(pd.ExcelFile(LRN_FILE).sheet_names)

# Prefer evidence_shortlist if present
lrn_sheets = pd.ExcelFile(LRN_FILE).sheet_names

if "evidence_shortlist" in lrn_sheets:
    s9_sheet = "evidence_shortlist"
elif "final_candidates" in lrn_sheets:
    s9_sheet = "final_candidates"
else:
    raise KeyError(
        f"Could not locate LRN evidence shortlist. Sheets: {lrn_sheets}"
    )

s9 = pd.read_excel(
    LRN_FILE,
    sheet_name=s9_sheet
).copy()

display(s9)
print("S9 rows:", len(s9))

S9_CSV = SUPP_TABLE_DIR / "TableS9_lrn_gm02_evidence_shortlist.csv"
S9_XLSX = SUPP_TABLE_DIR / "TableS9_lrn_gm02_evidence_shortlist.xlsx"

s9.to_csv(S9_CSV, index=False)

with pd.ExcelWriter(S9_XLSX, engine="openpyxl") as writer:
    s9.to_excel(writer, sheet_name="TableS9", index=False)


# ============================================================
# TABLE S10
# ============================================================

PROT_FILE = (
    CAND_DIR
    / "flyer_hartwig_prot03_gm20_final_evidence_summary.xlsx"
)

if not PROT_FILE.exists():
    raise FileNotFoundError(
        f"Missing frozen prot_03 workbook:\n{PROT_FILE}"
    )

print("\nprot_03 workbook sheets:")
print(pd.ExcelFile(PROT_FILE).sheet_names)

prot_sheets = pd.ExcelFile(PROT_FILE).sheet_names

if "curated_candidates" in prot_sheets:
    s10_sheet = "curated_candidates"
elif "final_candidates" in prot_sheets:
    s10_sheet = "final_candidates"
else:
    raise KeyError(
        f"Could not locate prot_03 curated candidate sheet. Sheets: {prot_sheets}"
    )

s10 = pd.read_excel(
    PROT_FILE,
    sheet_name=s10_sheet
).copy()

display(s10)
print("S10 rows:", len(s10))

S10_CSV = SUPP_TABLE_DIR / "TableS10_prot03_gm20_curated_candidates.csv"
S10_XLSX = SUPP_TABLE_DIR / "TableS10_prot03_gm20_curated_candidates.xlsx"

s10.to_csv(S10_CSV, index=False)

with pd.ExcelWriter(S10_XLSX, engine="openpyxl") as writer:
    s10.to_excel(writer, sheet_name="TableS10", index=False)

print("\nSaved:")
print(S9_CSV)
print(S9_XLSX)
print(S10_CSV)
print(S10_XLSX)